# Sanity Check Tokens



In [79]:
import itertools

from transformers import AutoTokenizer, PreTrainedTokenizer
from tqdm.auto import tqdm

def open_txt(path):
    with open(path, 'r') as file:
        words = [line.strip() for line in file] 
    return words

def save_txt(path, text):
    with open(path, 'w') as file:
        file.write(text)


---

## 1) Clean word list

Cleaning or sanitizing the word lists means checking out which of the uni-gram or bi-grams are not actually considered *words* but (for whatever reason) are still part of the word list.

In [80]:
# Generate all permutations of two letters from 'a' to 'z'
letters = [chr(i) for i in range(ord('a'), ord('z') + 1)]
permutations_2 = itertools.product(letters, repeat=2)
# Convert tuples to strings and print or store them
permutations_list_2 = [''.join(p) for p in permutations_2]
print(f"There are {len(permutations_list_2)} permutations of two letters.")

There are 676 permutations of two letters.


### NLTK

Using the NLTK list of words ~236k.
But we filter out all single letters as well as some of the 2-grams.

The word list of NLTK can be obtained and saved in a file by installing `nltk` and then running:
> ```Python
> import nltk
> nltk.download('words')
> from nltk.corpus import words
> text = '\n'.join(words.words())
> with open('words_nltk.txt', 'w') as file:
>         file.write(text)
> ```

In [78]:
path_nltk = "../src/delphi/eval/words_nltk.txt" 
words_nltk = open_txt(path_nltk)   
print("NLTK number of words", len(words_nltk))

letters_found, bigrams_found = list(), list()
for word in words_nltk:
    word = word.lower()
    if word in letters:
        letters_found.append(word)
    if word in permutations_list_2:
        bigrams_found.append(word)
print(f"There are {len(letters_found)} letters that are in the list of common words.")
print(f"There are {len(bigrams_found)} bigrams that are in the list of common words.")

# remove letters and bigrams that are not common words but present in the nltk word list
letters_to_be_removed = ['b', 'c', 'd', 'e', 'f', 'g', 'h', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
bigrams_to_be_removed = ['sa', 'hu', 'ba', 'ca', 'os', 'sh', 'ug', 'bu', 'ro', 'si', 'oe', 'yr', 'om', 'el', 'fe', 'se', 'wa', 'ji', 'og', 'da', 'zo', 'tu', 'ga', 'mo', 'yn', 'td', 'ea', 'us', 'st', 'fu', 'ar', 'al', 'ni', 'ly', 'pi', 'la', 'id', 'ka', 'ex', 'wu', 'ok', 'ym', 'on', 'un', 'ak', 'ae', 'no', 'hy', 'ko', 'ax', 'wi', 'za', 'gi', 'is', 'ab', 'ur', 'es', 'nu', 'ge', 'io', 'jo', 'em', 'er', 'lo', 'ai', 'vu', 'fi', 'ud', 're', 'od', 'aw', 'ra', 'ut', 'ye', 'pa', 'ya', 'xi', 'de', 'ti', 'wy', 'fo', 'ao', 'li', 'ie', 'wo', 'te', 'na', 'ow', 'pu', 'ma', 'po', 'yo', 'ne', 'ju', 'aa', 'ce', 'mu', 'lu', 'mi', 'ed', 'en', 'ox', 'mr', 'ha', 'bo', 'ta', 'eu', 'th', 'di', 'fa']
# iterate over the words in nltk
indices = list()
for i, word in enumerate(words_nltk):
    word = word.lower()
    if word in letters_to_be_removed:
        indices.append(i)
    if word in bigrams_to_be_removed:
        indices.append(i)
indices = list(set(indices)) # remove duplicates
print("Words to be removed:", len(indices))

words_nltk_cleaned = [word for i, word in enumerate(words_nltk) if i not in indices]
assert len(words_nltk_cleaned) == len(words_nltk) - len(indices)
print("NLTK number of words after cleaning", len(words_nltk_cleaned))


text = '\n'.join(words_nltk_cleaned)
save_txt('../src/delphi/eval/words_nltk_cleaned.txt', text)

NLTK number of words 236736
There are 54 letters that are in the list of common words.
There are 176 bigrams that are in the list of common words.
Words to be removed: 177
NLTK number of words after cleaning 236559


---

## 2) Compare tokens in tokenizer

First we load the tokenizer.

In [5]:
# Decode a sentence
def decode(tokenizer: PreTrainedTokenizer, token_ids: list[int]) -> str:
    return tokenizer.decode(token_ids, skip_special_tokens=True)

model = "delphi-suite/delphi-llama2-100k"
tokenizer = AutoTokenizer.from_pretrained(model)
vocabulary = tokenizer.get_vocab()
vocab_size = tokenizer.vocab_size
print("The vocab size is:", vocab_size)

The vocab size is: 4096


Now we check for the permutation that are in the tokenizer's vocabulary and in the words_nltk_cleaned

In [67]:
words_nltk_cleaned = open_txt('../src/delphi/eval/words_nltk_cleaned.txt')
# Check if the 2-, 3-letter permutations are in the vocabulary of tokenizer
print("In Vocab of tokenizer:")
count = 0
for p in permutations_list_2:
    if p in vocabulary:
        count += 1
print(f"\tThere are {count} / {len(permutations_list_2)} permutations of two letters")
print("In Vocab of words_nltk:")
count = 0
for p in permutations_list_2:
    if p in words_nltk:
        count += 1
print(f"\tThere are {count} / {len(permutations_list_2)} permutations of two letters")

In Vocab of tokenizer:
	There are 153 / 676 permutations of two letters
In Vocab of words_nltk:
	There are 121 / 676 permutations of two letters


In [72]:
# check how many tokens are in the nltk words
count = 0
words, not_words = list(), list()
# for every token in the tokenizer's vocabulary
for i in tqdm(range(vocab_size)):
    # get str representation
    token = decode(tokenizer, [i])
    # check if it is in the nltk words
    if token.strip().lower() in words_nltk_cleaned:
        count += 1
        words.append(token)
    else:
        not_words.append(token)
print(f"There are {count} / {len(vocabulary)} words in the vocabulary of the tokenizer.")

  0%|          | 0/4096 [00:00<?, ?it/s]

There are 2404 / 4096 words in the vocabulary of the tokenizer.


In [73]:
print("\n".join(not_words[:1000]))




 








	
























!
"
#
$
%
&
'
(
)
*
+
,
-
.
/
0
1
2
3
4
5
6
7
8
9
:
;
<
=
>
?
@
B
C
D
E
F
G
H
J
K
L
M
N
O
P
Q
R
S
T
U
V
W
X
Y
Z
[
\
]
^
_
`
b
c
d
e
f
g
h
j
k
l
m
n
o
p
q
r
s
t
u
v
w
x
y
z
{
|
}
~

�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
�
t
s
w
nd
ed
b
T
h
f
wa
re
ou
l
d
c
p
m
er
om
is
n
ar
im
on
sa
id
ll
S
ha
g
ot
en
an
le
"
ir
H
et
th
ig
il
pl
ow
ri
ver
ut
u
O
ith
Tim
pp
On
o
y
oo
r
ce
ld
st
ke
e
L
nt
ck
st
ve
B
on
happ
un
riend
ily
M
li
itt
se
ent
es
ould
mom
sh
ittle
Tom
."
ime
ch
ne
k
ound
named
bo
sm
!"
wanted
friends
ved
ht
ird
el
al
an
ome
ug
hel
wh
is
ue
lo
ter
ry
ind
ra
j
re
ur
se
ack
tre
ly
ood
ic
ard
?"
ec
played
gir
ro
hed
le
fr
ain
kn
ax
W
ul
Max
loved
cl
looked
oug
sp
sc
ful
bec
ong
ight
op
liked
J
la
fa
omet


In [77]:
'liked' in words_nltk

False